# Notebook de forecasting
Este notebook importa las librerías necesarias y carga los datos de inferencia para el análisis de forecasting.

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import holidays

In [14]:
# Cargar el archivo de inferencia en un DataFrame
inferencia_df = pd.read_csv('../data/raw/inferencia/ventas_2025_inferencia.csv')

# Verificar datos únicos
print("Datos de inferencia cargados:")
print(f"Forma: {inferencia_df.shape}")
print(f"Fechas únicas: {sorted(inferencia_df['fecha'].unique())}")
print(f"Meses únicos en fecha: {sorted(pd.to_datetime(inferencia_df['fecha']).dt.month.unique())}")
print(f"\nPrimeras filas:")
print(inferencia_df.head())

Datos de inferencia cargados:
Forma: (888, 13)
Fechas únicas: ['2025-10-25', '2025-10-26', '2025-10-27', '2025-10-28', '2025-10-29', '2025-10-30', '2025-10-31', '2025-11-01', '2025-11-02', '2025-11-03', '2025-11-04', '2025-11-05', '2025-11-06', '2025-11-07', '2025-11-08', '2025-11-09', '2025-11-10', '2025-11-11', '2025-11-12', '2025-11-13', '2025-11-14', '2025-11-15', '2025-11-16', '2025-11-17', '2025-11-18', '2025-11-19', '2025-11-20', '2025-11-21', '2025-11-22', '2025-11-23', '2025-11-24', '2025-11-25', '2025-11-26', '2025-11-27', '2025-11-28', '2025-11-29', '2025-11-30']
Meses únicos en fecha: [np.int32(10), np.int32(11)]

Primeras filas:
        fecha producto_id                            nombre categoria  \
0  2025-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1  2025-10-25    PROD_002              Adidas Ultraboost 23   Running   
2  2025-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3  2025-10-25    PROD_004  New Balance Fresh Foam X 1080v1

In [15]:
# Crear variables temporales útiles para predicción
# Aseguramos formato de fecha
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# Año, mes, semana, día, día de la semana, día del mes
inferencia_df['anio'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['semana'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['dia'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.weekday  # 0=lunes
inferencia_df['dia_semana_nombre'] = inferencia_df['fecha'].dt.day_name(locale='es')

# Mes-semana, semana-día, mes-día
inferencia_df['mes_semana'] = inferencia_df['mes'].astype(str) + '-' + inferencia_df['semana'].astype(str)
inferencia_df['semana_dia'] = inferencia_df['semana'].astype(str) + '-' + inferencia_df['dia_semana'].astype(str)
inferencia_df['mes_dia'] = inferencia_df['mes'].astype(str) + '-' + inferencia_df['dia'].astype(str)

# ¿Es fin de semana?
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5, 6])

# Feriados Colombia
dias_feriados = holidays.country_holidays('CO', years=inferencia_df['anio'].unique())
inferencia_df['es_feriado'] = inferencia_df['fecha'].isin(dias_feriados)

# Black Friday (cuarto viernes de noviembre)
def es_blackfriday(fecha):
    if fecha.month == 11 and fecha.weekday() == 4:
        fridays = [d for d in pd.date_range(start=fecha.replace(day=1), end=fecha.replace(day=30)) if d.weekday() == 4]
        if len(fridays) >= 4 and fecha == fridays[3]:
            return True
    return False

inferencia_df['es_blackfriday'] = inferencia_df['fecha'].apply(es_blackfriday)

# Cyber Monday (primer lunes después de Black Friday)
def es_cybermonday(fecha):
    if fecha.month == 11 or fecha.month == 12:
        year = fecha.year
        nov = pd.date_range(start=f'{year}-11-01', end=f'{year}-11-30', freq='D')
        fridays = nov[nov.weekday == 4]
        if len(fridays) >= 4:
            black_friday = fridays[3]
            cyber_monday = black_friday + pd.Timedelta(days=3)
            return fecha == cyber_monday
    return False

inferencia_df['es_cybermonday'] = inferencia_df['fecha'].apply(es_cybermonday)

# Variables adicionales útiles
inferencia_df['es_principio_mes'] = inferencia_df['dia'] <= 3
inferencia_df['es_final_mes'] = inferencia_df['dia'] >= (inferencia_df['fecha'] + pd.offsets.MonthEnd(0)).dt.day - 2
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['es_lunes'] = inferencia_df['dia_semana'] == 0
inferencia_df['es_viernes'] = inferencia_df['dia_semana'] == 4

print("Variables temporales creadas")
print(f"Forma del dataframe: {inferencia_df.shape}")
print(f"Primeras filas:\n{inferencia_df.head()}")

Variables temporales creadas
Forma del dataframe: (888, 31)
Primeras filas:
       fecha producto_id                            nombre categoria  \
0 2025-10-25    PROD_001          Nike Air Zoom Pegasus 40   Running   
1 2025-10-25    PROD_002              Adidas Ultraboost 23   Running   
2 2025-10-25    PROD_003               Asics Gel Nimbus 25   Running   
3 2025-10-25    PROD_004  New Balance Fresh Foam X 1080v12   Running   
4 2025-10-25    PROD_005                Nike Dri-FIT Miler   Running   

         subcategoria  precio_base  es_estrella  unidades_vendidas  \
0  Zapatillas Running          115         True               26.0   
1  Zapatillas Running          135         True               27.0   
2  Zapatillas Running           85        False                5.0   
3  Zapatillas Running           75        False                3.0   
4        Ropa Running           35        False                3.0   

   precio_venta  ingresos  ...  mes_dia  es_fin_semana  es_feriado  \


In [16]:
# Crear lags y medias móviles de unidades vendidas
def crear_lags_y_media(df_):
    df_ = df_.sort_values('fecha').copy()
    for lag in range(1, 8):
        df_[f'unidades_vendidas_lag{lag}'] = df_['unidades_vendidas'].shift(lag)
    df_['unidades_vendidas_mm7'] = df_['unidades_vendidas'].rolling(window=7, min_periods=1).mean()
    return df_

# Aplicar lags por año
frames = []
for anio, grupo in inferencia_df.groupby('anio'):
    grupo_lags = crear_lags_y_media(grupo)
    frames.append(grupo_lags)

# Concatenar los dataframes
if len(frames) > 0:
    inferencia_df = pd.concat(frames, ignore_index=True).reset_index(drop=True)
else:
    print("Advertencia: No se encontraron datos para agrupar por año")

# Nota: Los primeros registros de octubre tendrán NaN en los lags, que es esperado
# Los registros de noviembre tendrán lags disponibles desde octubre

print("Lags y medias móviles creadas")
print(f"Forma del dataframe: {inferencia_df.shape}")
print(f"Primeros registros con lags (algunos pueden tener NaN):")
print(inferencia_df[['fecha', 'unidades_vendidas', 'unidades_vendidas_lag1', 'unidades_vendidas_lag7', 'unidades_vendidas_mm7']].head(15))

Lags y medias móviles creadas
Forma del dataframe: (888, 39)
Primeros registros con lags (algunos pueden tener NaN):
        fecha  unidades_vendidas  unidades_vendidas_lag1  \
0  2025-10-25               26.0                     NaN   
1  2025-10-25                1.0                    26.0   
2  2025-10-25                1.0                     1.0   
3  2025-10-25                3.0                     1.0   
4  2025-10-25               10.0                     3.0   
5  2025-10-25                1.0                    10.0   
6  2025-10-25                3.0                     1.0   
7  2025-10-25                3.0                     3.0   
8  2025-10-25                5.0                     3.0   
9  2025-10-25                6.0                     5.0   
10 2025-10-25                5.0                     6.0   
11 2025-10-25                1.0                     5.0   
12 2025-10-25                3.0                     1.0   
13 2025-10-25                7.0           

In [17]:
# Crear variable porcentaje de descuento respecto al precio base
inferencia_df['porc_descuento'] = ((inferencia_df['precio_base'] - inferencia_df['precio_venta']) / inferencia_df['precio_base']) * 100

# Crear variable precio_competencia (promedio de Amazon, Decathlon y Deporvillage)
inferencia_df['precio_competencia'] = inferencia_df[['Amazon', 'Decathlon', 'Deporvillage']].mean(axis=1)

# Crear variable ratioprecio (precio_venta / precio_competencia)
inferencia_df['ratioprecio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

# Eliminar variables de la competencia
inferencia_df = inferencia_df.drop(['Amazon', 'Decathlon', 'Deporvillage'], axis=1)

print("Variables de descuento y ratio de precio creadas")
print(f"Forma del dataframe: {inferencia_df.shape}")
print(f"Primeras filas con nuevas variables:\n{inferencia_df[['fecha', 'porc_descuento', 'precio_competencia', 'ratioprecio']].head()}")

Variables de descuento y ratio de precio creadas
Forma del dataframe: (888, 39)
Primeras filas con nuevas variables:
       fecha  porc_descuento  precio_competencia  ratioprecio
0 2025-10-25        1.626087          102.573333     1.102918
1 2025-10-25        1.720000           49.996667     0.982866
2 2025-10-25       -0.657143           33.860000     1.040461
3 2025-10-25       -1.850000           19.846667     1.026369
4 2025-10-25       -0.092308          122.116667     1.065538


In [18]:
# Crear copias de variables categóricas con sufijo _h
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# One hot encoding sobre las copias
categorias_ohe = ['nombre_h', 'categoria_h', 'subcategoria_h']
inferencia_df = pd.get_dummies(inferencia_df, columns=categorias_ohe, prefix=categorias_ohe, prefix_sep='_', dtype=int)

print("One-hot encoding aplicado")
print(f"Forma del dataframe: {inferencia_df.shape}")
print(f"Nuevas columnas creadas por one-hot encoding:")
ohe_cols = [col for col in inferencia_df.columns if any(col.startswith(f"{prefix}_") for prefix in categorias_ohe)]
print(f"Total de columnas one-hot: {len(ohe_cols)}")
print(f"Primeras columnas: {ohe_cols[:10]}")

One-hot encoding aplicado
Forma del dataframe: (888, 83)
Nuevas columnas creadas por one-hot encoding:
Total de columnas one-hot: 44
Primeras columnas: ['nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900', 'nombre_h_Domyos Kit Mancuernas 20kg', 'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad']


In [19]:
# Filtrar para dejar solo registros de noviembre (eliminar octubre)
print(f"Meses únicos antes del filtrado: {sorted(inferencia_df['mes'].unique())}")
print(f"Registros por mes antes del filtrado:")
print(inferencia_df['mes'].value_counts().sort_index())

# Filtrar solo noviembre (mes == 11)
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].reset_index(drop=True)

print(f"\nDespués del filtrado a noviembre:")
print(f"Forma del dataframe: {inferencia_df.shape}")
print(f"Rango de fechas: {inferencia_df['fecha'].min()} a {inferencia_df['fecha'].max()}")
print(f"Primeras filas después del filtrado:\n{inferencia_df[['fecha', 'producto_id', 'nombre', 'unidades_vendidas']].head(10)}")

Meses únicos antes del filtrado: [np.int32(10), np.int32(11)]
Registros por mes antes del filtrado:
mes
10    168
11    720
Name: count, dtype: int64

Después del filtrado a noviembre:
Forma del dataframe: (720, 83)
Rango de fechas: 2025-11-01 00:00:00 a 2025-11-30 00:00:00
Primeras filas después del filtrado:
       fecha producto_id                    nombre  unidades_vendidas
0 2025-11-01    PROD_014   Sveltus Kettlebell 12kg                NaN
1 2025-11-01    PROD_023          Liforme Yoga Pad                NaN
2 2025-11-01    PROD_022  Gaiam Premium Yoga Block                NaN
3 2025-11-01    PROD_021      Manduka PRO Yoga Mat                NaN
4 2025-11-01    PROD_020             Quechua MH500                NaN
5 2025-11-01    PROD_019        Merrell Moab 2 GTX                NaN
6 2025-11-01    PROD_018     Columbia Silver Ridge                NaN
7 2025-11-01    PROD_017   The North Face Borealis                NaN
8 2025-11-01    PROD_016             Trek Marlin 7        

In [20]:
# Guardar el dataframe transformado
guardado_path = '../data/processed/inferencia_df_transformado.csv'
inferencia_df.to_csv(guardado_path, index=False)

print(f"✓ DataFrame transformado guardado en: {guardado_path}")
print(f"\nResumen final:")
print(f"- Registros: {len(inferencia_df)}")
print(f"- Variables: {inferencia_df.shape[1]}")
print(f"- Mes: {inferencia_df['mes'].unique()[0] if len(inferencia_df['mes'].unique()) > 0 else 'N/A'} (11=Noviembre)")
print(f"- Año: {inferencia_df['anio'].unique()[0] if len(inferencia_df['anio'].unique()) > 0 else 'N/A'}")
print(f"- Rango de fechas: {inferencia_df['fecha'].min()} a {inferencia_df['fecha'].max()}")
print(f"\nColumnas del dataframe transformado:")
print(inferencia_df.columns.tolist())
print(f"\nDataframe listo para usar con el modelo de forecasting")

✓ DataFrame transformado guardado en: ../data/processed/inferencia_df_transformado.csv

Resumen final:
- Registros: 720
- Variables: 83
- Mes: 11 (11=Noviembre)
- Año: 2025
- Rango de fechas: 2025-11-01 00:00:00 a 2025-11-30 00:00:00

Columnas del dataframe transformado:
['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'anio', 'mes', 'semana', 'dia', 'dia_semana', 'dia_semana_nombre', 'mes_semana', 'semana_dia', 'mes_dia', 'es_fin_semana', 'es_feriado', 'es_blackfriday', 'es_cybermonday', 'es_principio_mes', 'es_final_mes', 'trimestre', 'es_lunes', 'es_viernes', 'unidades_vendidas_lag1', 'unidades_vendidas_lag2', 'unidades_vendidas_lag3', 'unidades_vendidas_lag4', 'unidades_vendidas_lag5', 'unidades_vendidas_lag6', 'unidades_vendidas_lag7', 'unidades_vendidas_mm7', 'porc_descuento', 'precio_competencia', 'ratioprecio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nom

In [22]:
inferencia_df.columns

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'anio', 'mes', 'semana', 'dia', 'dia_semana',
       'dia_semana_nombre', 'mes_semana', 'semana_dia', 'mes_dia',
       'es_fin_semana', 'es_feriado', 'es_blackfriday', 'es_cybermonday',
       'es_principio_mes', 'es_final_mes', 'trimestre', 'es_lunes',
       'es_viernes', 'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7', 'porc_descuento',
       'precio_competencia', 'ratioprecio',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
   